Sử dụng mô hình Embeddings 

In [2]:
# Thêm thư viện cho mô hình all-MinLM-L6-v2
import pandas as pd
import numpy as np
import pickle
import torch
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/home/vophilong/Documents/Deep_learning/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mô hình all-MinLM-L6-v2 dùng phương pháp Lazy Learning (học lười). Nó không trải qua nhiều vòng lặp (Epochs) để cập nhật trọng số, mà chỉ đơn giản là lấy nguyên tập Train đem cất vào kho (Vector Database).

Tập val sẽ dùng cho các mô hình T5, BART, Phi-3, Qwen

In [3]:
train_df = pd.read_csv('/home/vophilong/Documents/Deep_learning/DoAn_Email/dataset/dataset_split/train.csv').dropna(subset=['Input_Text', 'Output_Text']).reset_index(drop=True)
test_df = pd.read_csv('/home/vophilong/Documents/Deep_learning/DoAn_Email/dataset/dataset_split/test.csv').dropna(subset=['Input_Text', 'Output_Text']).reset_index(drop=True)

print(f"Số lượng Train: {len(train_df)} | Số lượng Test: {len(test_df)}")


Số lượng Train: 11069 | Số lượng Test: 1384


In [4]:
print("Đang khởi tạo mô hình Embedding (all-MiniLM-L6-v2)...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
retriever_model = SentenceTransformer('all-MiniLM-L6-v2').to(device) # Sử dụng mô hình all-MiniLM-L6-v2 để tạo vector embeddings

print("Đang mã hóa tập Train (Tạo Vector Database)...")
train_embeddings = retriever_model.encode(train_df['Input_Text'].tolist(), # Tạo vector embeddings cho tập Train
                                          batch_size=64, 
                                          show_progress_bar=True,  # Hiển thị thanh tiến trình
                                          convert_to_tensor=True) # Chuyển đổi kết quả thành tensor để tối ưu hóa hiệu suất trên GPU nếu có

Đang khởi tạo mô hình Embedding (all-MiniLM-L6-v2)...


/home/vophilong/Documents/Deep_learning/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10197.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang mã hóa tập Train (Tạo Vector Database)...


Batches: 100%|██████████| 173/173 [01:42<00:00,  1.68it/s]


### INFERENCE & ĐÁNH GIÁ TRÊN TẬP TEST

In [5]:
print("Đang mã hóa tập Test và Truy xuất câu trả lời...")
test_embeddings = retriever_model.encode(test_df['Input_Text'].tolist(), 
                                         batch_size=64, 
                                         show_progress_bar=True, 
                                         convert_to_tensor=True) 

Đang mã hóa tập Test và Truy xuất câu trả lời...


Batches: 100%|██████████| 22/22 [00:12<00:00,  1.77it/s]


In [6]:
# Tính Cosine Similarity
cos_scores = cosine_similarity(test_embeddings.cpu().numpy(), train_embeddings.cpu().numpy())

In [ ]:
# Lấy index và điểm số của câu giống nhất
best_match_indices = np.argmax(cos_scores, axis=1) # Index của câu trả lời tốt nhất trong tập Train
confidence_scores = np.max(cos_scores, axis=1) # Điểm tự tin (Similarity)

In [8]:
# Lấy dự đoán
predictions = [train_df['Output_Text'].iloc[idx] for idx in best_match_indices] # Lấy câu trả lời thực tế từ tập Train dựa trên index tìm được
references = test_df['Output_Text'].tolist() # Tính BLEU Score

In [ ]:
print("Đang tính toán ROUGE, BLEU và BERTScore (Quá trình này có thể mất vài phút)...")
# Load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('sacrebleu')
bertscore = evaluate.load('bertscore')
# Compute
rouge_res = rouge.compute(predictions=predictions, references=references)
bleu_res = bleu.compute(predictions=predictions, references=[[ref] for ref in references])
# BERTScore sử dụng mô hình distilbert để tính toán cho lẹ
bert_res = bertscore.compute(predictions=predictions, references=references, lang="en", model_type="distilbert-base-uncased")

# Gom kết quả
results = {
    "ROUGE-1": rouge_res['rouge1'],
    "ROUGE-2": rouge_res['rouge2'],
    "ROUGE-L": rouge_res['rougeL'],
    "BLEU": bleu_res['score'] / 100, # Đưa BLEU về scale 0-1 cho dễ vẽ biểu đồ
    "BERTScore (F1)": np.mean(bert_res['f1'])
}

Đang tính toán ROUGE, BLEU và BERTScore (Quá trình này có thể mất vài phút)...


ImportError: To be able to use evaluate-metric/bertscore, you need to install the following dependencies['bert_score'] using 'pip install bert_score' for instance'